In [ ]:
# 02 — Machine Learning

This notebook trains and evaluates models on spectral features computed from
`.Q00` conductivity fluctuation measurements.

It uses the combined dataset produced by `src/signal_processing.py` and
reproduces the pipeline implemented in `src/train_models.py`.

Stable Bi, Cr, and Fe films are used for supervised material recognition.
XXX reference films and irradiated films are excluded from training.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = BASE_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

DATASET_PATH = BASE_DIR / "data" / "processed" / "conductivity_features.csv"
FIGURES_DIR = BASE_DIR / "results" / "figures"
METRICS_DIR = BASE_DIR / "results" / "metrics"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_LABEL = "XXX"
MATERIAL_ORDER = ["BI", "CR", "FE"]
RANDOM_STATE = 42

MATERIAL_NAMES = {
    "BI": "Bismuth (Bi)",
    "CR": "Chromium (Cr)",
    "FE": "Iron (Fe)",
    "XXX": "Reference film",
}

MATERIAL_COLORS = {
    "BI": "#4C72B0",
    "CR": "#DD8452",
    "FE": "#55A868",
    "XXX": "#000000",
}

print("Dataset:", DATASET_PATH)
print("Figures:", FIGURES_DIR)
print("Metrics:", METRICS_DIR)

In [ ]:
## Load the combined feature dataset

The dataset must first be created by running:

```bash
python src/signal_processing.py
```

In [ ]:
df = pd.read_csv(DATASET_PATH)

print(f"Rows: {len(df)}")
print(f"Files: {df['file'].nunique()}")
print(f"Columns: {list(df.columns)}")

df.head()

In [ ]:
## Dataset overview

Only stable, non-irradiated Bi, Cr, and Fe films are used for supervised
learning. The `used_for_training` column is produced by
`src/signal_processing.py`.

In [ ]:
df["material"].value_counts()

In [ ]:
df.groupby(["material", "irradiated", "used_for_training"]).size()

In [ ]:
## Prepare training data

The training set contains only stable films marked as `used_for_training`.
Reference films (`XXX`) and irradiated films are excluded.

In [ ]:
def prepare_training_data(df: pd.DataFrame):
    df_train = df[
        (df["used_for_training"] == True)
        & (df["material"] != REFERENCE_LABEL)
        & (df["irradiated"] == False)
    ].copy()

    X = df_train[["gamma", "log_energy"]].copy()
    y = df_train["material"].copy()

    return X, y, df_train


X, y, df_train = prepare_training_data(df)

print(f"Training samples: {len(X)}")
print("Class distribution:")
print(y.value_counts())

X.head()

In [ ]:
## Train-test split

The split is stratified by material class and uses a fixed random seed.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

print("\nTrain class distribution:")
print(y_train.value_counts())

print("\nTest class distribution:")
print(y_test.value_counts())

In [ ]:
## Feature standardization

The scaler is fitted only on the training set and then applied to the test set.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
).describe()

In [ ]:
## K-Means clustering

K-Means is fitted on the standardized training features. The number of
clusters equals the number of material classes.

In [ ]:
n_clusters = y.nunique()

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=RANDOM_STATE,
    n_init=20,
)

cluster_labels = kmeans.fit_predict(X_train_scaled)

silhouette = silhouette_score(X_train_scaled, cluster_labels)

print(f"Number of clusters: {n_clusters}")
print(f"Silhouette score: {silhouette:.4f}")

In [ ]:
def plot_clusters(X_scaled, labels, path):
    plt.figure(figsize=(10, 6.5))

    scatter = plt.scatter(
        X_scaled[:, 0],
        X_scaled[:, 1],
        c=labels,
        cmap="viridis",
        s=25,
        alpha=0.75,
        edgecolor="black",
        linewidth=0.2,
    )

    plt.xlabel("Spectral exponent γ (standardized)", fontsize=12)
    plt.ylabel("Log10 spectral energy (standardized)", fontsize=12)
    plt.title("K-Means Clustering of Stable Film Noise Features", fontsize=14)
    plt.colorbar(scatter, label="Cluster ID")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()


plot_clusters(
    X_train_scaled,
    cluster_labels,
    FIGURES_DIR / "kmeans_clusters.png",
)

In [ ]:
## K-Nearest Neighbors

KNN is trained on standardized spectral features and evaluated on the held-out
test set.

In [ ]:
knn = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
)

knn.fit(X_train_scaled, y_train)

knn_pred = knn.predict(X_test_scaled)
knn_accuracy = accuracy_score(y_test, knn_pred)

print(f"KNN accuracy: {knn_accuracy * 100:.2f}%")
print(
    classification_report(
        y_test,
        knn_pred,
        labels=MATERIAL_ORDER,
        target_names=[MATERIAL_NAMES[label] for label in MATERIAL_ORDER],
        zero_division=0,
    )
)

In [ ]:
## Neural network classifier

A multilayer perceptron is trained on the same standardized features.

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    max_iter=1000,
    early_stopping=True,
    random_state=RANDOM_STATE,
)

mlp.fit(X_train_scaled, y_train)

mlp_pred = mlp.predict(X_test_scaled)
mlp_accuracy = accuracy_score(y_test, mlp_pred)

print(f"Neural network accuracy: {mlp_accuracy * 100:.2f}%")
print(
    classification_report(
        y_test,
        mlp_pred,
        labels=MATERIAL_ORDER,
        target_names=[MATERIAL_NAMES[label] for label in MATERIAL_ORDER],
        zero_division=0,
    )
)

In [ ]:
## Neural network confusion matrix

In [ ]:
def plot_confusion_matrix(y_test, y_pred, labels, path):
    readable_labels = [MATERIAL_NAMES[label] for label in labels]
    cm = confusion_matrix(y_test, y_pred, labels=labels)

    plt.figure(figsize=(9, 7))

    ax = sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=readable_labels,
        yticklabels=readable_labels,
        cbar_kws={"label": "Number of test samples"},
        linewidths=0.5,
        linecolor="white",
    )

    ax.set_xlabel("Predicted material")
    ax.set_ylabel("True material")
    ax.set_title("Neural Network Classification\nof Stable Thin-Film Materials")

    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()


plot_confusion_matrix(
    y_test=y_test,
    y_pred=mlp_pred,
    labels=MATERIAL_ORDER,
    path=FIGURES_DIR / "neural_network_confusion_matrix.png",
)

In [ ]:
## Apply the trained model to irradiated films

Irradiated films were not used for training. The trained neural network is
applied to them to obtain predicted material labels and confidence values.

In [ ]:
df_irradiated = df[df["irradiated"] == True].copy()

if df_irradiated.empty:
    print("No irradiated films found.")
else:
    X_irradiated = df_irradiated[["gamma", "log_energy"]].copy()
    X_irradiated_scaled = scaler.transform(X_irradiated)

    df_irradiated["predicted_material"] = mlp.predict(X_irradiated_scaled)

    if hasattr(mlp, "predict_proba"):
        probabilities = mlp.predict_proba(X_irradiated_scaled)
        df_irradiated["prediction_confidence"] = probabilities.max(axis=1)

    df_irradiated[
        [
            "file",
            "material",
            "gamma",
            "log_energy",
            "predicted_material",
            "prediction_confidence",
        ]
    ].head()

In [ ]:
## File-level comparison of spectral features

The plots compare stable and irradiated films at the file level.

In [ ]:
def plot_file_level_comparison(df, feature, ylabel, filename):
    summary = (
        df[df["material"] != REFERENCE_LABEL]
        .groupby(["file", "material", "irradiated"], as_index=False)
        .agg(
            mean_value=(feature, "mean"),
            std_value=(feature, "std"),
            n_samples=(feature, "size"),
        )
    )

    summary["label"] = summary["file"] + "\n(" + summary["material"] + ")"

    summary = summary.sort_values(
        ["material", "irradiated", "file"],
        ascending=[True, True, True],
    )

    plt.figure(figsize=(13, 7))

    for material in MATERIAL_ORDER:
        material_summary = summary[summary["material"] == material]

        for _, row in material_summary.iterrows():
            marker = "o" if not row["irradiated"] else "X"
            edgecolor = (
                "black" if row["irradiated"] else MATERIAL_COLORS[material]
            )
            linewidth = 1.2 if row["irradiated"] else 0.8

            plt.errorbar(
                row["label"],
                row["mean_value"],
                yerr=row["std_value"],
                fmt=marker,
                markersize=10,
                color=MATERIAL_COLORS[material],
                markeredgecolor=edgecolor,
                markeredgewidth=linewidth,
                capsize=5,
                elinewidth=1.5,
                label=(
                    f"{MATERIAL_NAMES[material]} — "
                    + ("irradiated" if row["irradiated"] else "stable")
                ),
            )

    plt.ylabel(ylabel)
    plt.xlabel("Measurement file")
    plt.title(
        f"File-Level Comparison of {ylabel}\nStable vs Irradiated Films"
    )
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.grid(True, axis="y", alpha=0.3)

    handles, labels = plt.gca().get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    plt.legend(unique.values(), unique.keys(), fontsize=9)

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


plot_file_level_comparison(
    df=df,
    feature="gamma",
    ylabel="Spectral exponent γ",
    filename="file_level_gamma_comparison.png",
)

plot_file_level_comparison(
    df=df,
    feature="log_energy",
    ylabel="Log10 spectral energy",
    filename="file_level_energy_comparison.png",
)

In [ ]:
## Feature distributions by file

In [ ]:
def plot_feature_boxplots(df, feature, ylabel, filename):
    plot_df = df[df["material"] != REFERENCE_LABEL].copy()
    plot_df["file_label"] = (
        plot_df["file"] + "\n(" + plot_df["material"] + ")"
    )

    file_order = (
        plot_df[["file", "material", "irradiated"]]
        .drop_duplicates()
        .sort_values(["material", "irradiated", "file"])
    )

    ordered_labels = [
        f"{row['file']}\n({row['material']})"
        for _, row in file_order.iterrows()
    ]

    plt.figure(figsize=(15, 7))

    sns.boxplot(
        data=plot_df,
        x="file_label",
        y=feature,
        hue="irradiated",
        order=ordered_labels,
        palette={False: "#B0BEC5", True: "#FFB74D"},
        showfliers=False,
        width=0.7,
    )

    plt.ylabel(ylabel)
    plt.xlabel("Measurement file")
    plt.title(
        f"Distribution of {ylabel}\n"
        "by Measurement File and Irradiation State"
    )
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend(title="Film state")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


plot_feature_boxplots(
    df=df,
    feature="gamma",
    ylabel="Spectral exponent γ",
    filename="gamma_boxplots_by_file.png",
)

plot_feature_boxplots(
    df=df,
    feature="log_energy",
    ylabel="Log10 spectral energy",
    filename="energy_boxplots_by_file.png",
)

In [ ]:
## Save metrics and irradiated-film predictions

In [ ]:
metrics = pd.DataFrame(
    [
        {
            "model": "K-Means",
            "metric": "silhouette_score",
            "value": silhouette,
        },
        {
            "model": "KNN",
            "metric": "accuracy",
            "value": knn_accuracy,
        },
        {
            "model": "Neural network",
            "metric": "accuracy",
            "value": mlp_accuracy,
        },
    ]
)

metrics_path = METRICS_DIR / "model_metrics.csv"
metrics.to_csv(metrics_path, index=False)

print(metrics)
print(f"Metrics saved: {metrics_path}")

In [ ]:
if not df_irradiated.empty:
    predictions_path = METRICS_DIR / "irradiated_film_predictions.csv"
    df_irradiated.to_csv(predictions_path, index=False)
    print(f"Irradiated film predictions saved: {predictions_path}")
else:
    print("No irradiated film predictions were saved.")

In [ ]:
## Result files

The notebook creates the same main outputs as `src/train_models.py`:

- `results/figures/kmeans_clusters.png`
- `results/figures/neural_network_confusion_matrix.png`
- `results/figures/file_level_gamma_comparison.png`
- `results/figures/file_level_energy_comparison.png`
- `results/figures/gamma_boxplots_by_file.png`
- `results/figures/energy_boxplots_by_file.png`
- `results/metrics/model_metrics.csv`
- `results/metrics/irradiated_film_predictions.csv`